In [ ]:
from __future__ import annotations

import math
import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import ImageFile
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
EARLY_STOPPING_PATIENCE = 10

# input flag 
# "rgb_baseline" | "dynamic" | "static_ring" | "ring_ones"
INPUT_MODE = "dynamic" 

# settings (ignored when INPUT_MODE == "rgb_baseline" (original freqnet))
MID_HIGH_CUTOFF = 0.20  
NUM_BANDS = 8           
SELECTOR_HIDDEN = 32    

DATASET_HANDLE = "nocturnalnerd/balanced-dataset-701020"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_TAG = INPUT_MODE if INPUT_MODE == "rgb_baseline" else f"{INPUT_MODE}_c{MID_HIGH_CUTOFF:.2f}"
RUN_TAG = f"{RUN_TAG}_seed{SEED}"
CHECKPOINT_PATH = WORK_DIR / f"best_freqnet_{RUN_TAG}.pt"
EPOCH_CHECKPOINT_DIR = WORK_DIR / f"epoch_checkpoints_{RUN_TAG}"
EPOCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORK_DIR / "FreqNet-DeepfakeDetection"
REPO_URL = "https://github.com/chuangchuangtan/FreqNet-DeepfakeDetection.git"


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def resolve_dataset_root() -> Path:
    explicit_root = os.environ.get("DATASET_ROOT")
    if explicit_root:
        explicit_path = Path(explicit_root)
        if explicit_path.exists():
            return explicit_path
        raise FileNotFoundError(f"DATASET_ROOT={explicit_path} does not exist.")

    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub

    downloaded_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    direct = downloaded_root / "dataset_result_70_10_20_balanced"
    if direct.exists():
        return direct
    discovered = [p for p in downloaded_root.rglob("dataset_result_70_10_20_balanced") if p.is_dir()]
    if len(discovered) == 1:
        return discovered[0]
    if all((downloaded_root / split).is_dir() for split in ("train", "validation", "test")):
        return downloaded_root
    raise FileNotFoundError(f"Could not locate dataset_result_balanced under {downloaded_root}.")


seed_everything()
DATASET_ROOT = resolve_dataset_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset:", DATASET_ROOT)
print("Input mode:", INPUT_MODE)
print("Checkpoint target:", CHECKPOINT_PATH)


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

freqnet_source = REPO_DIR / "networks" / "freqnet.py"
source_text = freqnet_source.read_text(encoding="utf-8")
patched_text = source_text.replace(".cuda()", "")
if patched_text != source_text:
    freqnet_source.write_text(patched_text, encoding="utf-8")

sys.path.insert(0, str(REPO_DIR))
from networks.freqnet import FreqNet  # noqa: E402


class FakePositiveImageFolder(ImageFolder):
    """ImageFolder that always returns real=0 and fake=1 plus the image path."""

    def __init__(self, root: Path, transform=None):
        super().__init__(str(root), transform=transform)
        if set(self.classes) != {"fake", "real"}:
            raise ValueError(f"Expected class folders {{'real', 'fake'}} at {root}; found {self.classes}.")

    def __getitem__(self, index: int):
        image, original_target = super().__getitem__(index)
        path, _ = self.samples[index]
        class_name = self.classes[original_target]
        label = torch.tensor(float(class_name == "fake"), dtype=torch.float32)
        return image, label, path


train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
    ]
)
eval_transform = transforms.Compose(
    [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True), transforms.ToTensor()]
)

train_dataset = FakePositiveImageFolder(DATASET_ROOT / "train", train_transform)
val_dataset = FakePositiveImageFolder(DATASET_ROOT / "validation", eval_transform)
test_dataset = FakePositiveImageFolder(DATASET_ROOT / "test", eval_transform)


def class_counts(dataset: FakePositiveImageFolder) -> dict[str, int]:
    counts = Counter(dataset.classes[target] for _, target in dataset.samples)
    return {name: counts[name] for name in ("real", "fake")}


for split_name, split_dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    print(f"{split_name:10s} {class_counts(split_dataset)}")

loader_generator = torch.Generator().manual_seed(SEED)
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None, generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

class AmpSafeFreqNet(FreqNet):
    def __init__(self):
        super().__init__()
        with torch.no_grad():
            for weight, bias in (
                (self.weight1, self.bias1), (self.weight2, self.bias2),
                (self.weight3, self.bias3), (self.weight4, self.bias4),
            ):
                nn.init.kaiming_normal_(weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(bias)

    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            _, _, h, w = x.shape
            mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
            mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
            x = x * mask[None, None, :, :]
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def hfreqC(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft(x, dim=1, norm="ortho")
            x = torch.fft.fftshift(x, dim=1)
            _, c, _, _ = x.shape
            mask = torch.ones(c, device=x.device, dtype=x.real.dtype)
            mask[c // 2 - c // scale : c // 2 + c // scale] = 0.0
            x = x * mask[None, :, None, None]
            x = torch.fft.ifftshift(x, dim=1)
            x = torch.fft.ifft(x, dim=1, norm="ortho").real
            return F.relu(x, inplace=False)

    def _frequency_conv(self, x: torch.Tensor, real_conv: nn.Module, imag_conv: nn.Module) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            x = torch.complex(real_conv(x.real), imag_conv(x.imag))
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def _raw_projection(self, x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, stride: int) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = F.conv2d(x.float(), weight.float(), bias.float(), stride=stride, padding=0)
            return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight1, self.bias1, stride=1)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv1, self.imagconv1)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight2, self.bias2, stride=2)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv2, self.imagconv2)

        x = self.maxpool(x)
        x = self.layer1(x)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight3, self.bias3, stride=1)
        x = self._frequency_conv(x, self.realconv3, self.imagconv3)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight4, self.bias4, stride=2)
        x = self._frequency_conv(x, self.realconv4, self.imagconv4)

        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x)


class DynamicAmpSafeFreqNet(AmpSafeFreqNet):
    """

    The original first step (hfreqWH, scale=4) is
        FFT -> zero a fixed central square -> IFFT -> ReLU.
    Here it is
        FFT -> multiply by a band mask -> IFFT -> ReLU,
    with the FFT / shift / IFFT / real-part / ReLU / FP32 handling copied line by
    line from AmpSafeFreqNet.hfreqWH. Only the mask differs:

      "dynamic" : one weight per radial band, predicted per image from the
                  within-image-normalised mean log-magnitude of each band (MLP).
      "static"  : same rings, one learned weight per band shared by all images.
      "ones"    : same rings, every weight = 1 (a plain radial high-pass).
      "freqnet" : the original fixed square mask, run through this code path
                  (used only by the regression check below).

    Every later layer (hfreqC, frequency convs, residual layers, later hfreqWH
    calls) is inherited unchanged. super().__init__() runs first, so the backbone
    weights get the same RNG stream -- and therefore the same initial values --
    as the original model for a given seed.
    """

    MODES = ("dynamic", "static", "ones", "freqnet")

    def __init__(self, first_stage_mode: str = "dynamic", cutoff: float = 0.20, num_bands: int = 8,
                 hidden_dim: int = 32, image_size: int = 256):
        super().__init__()
        if first_stage_mode not in self.MODES:
            raise ValueError(f"first_stage_mode must be one of {self.MODES}")
        self.first_stage_mode = first_stage_mode
        self.cutoff = float(cutoff)
        self.num_bands = int(num_bands)
        self._stem_pending = False

        if first_stage_mode == "dynamic":
            self.dynamic_selector = nn.Sequential(
                nn.Linear(num_bands, hidden_dim), nn.ReLU(inplace=True),
                nn.Linear(hidden_dim, num_bands), nn.Sigmoid(),
            )
        elif first_stage_mode == "static":
            self.static_band_logits = nn.Parameter(torch.zeros(num_bands))  # sigmoid(0) = 0.5, like the MLP at init

        band_plus1, membership, counts = self._build_band_geometry(image_size, image_size)
        self.register_buffer("_band_plus1", band_plus1, persistent=False)   # 0 = suppressed, k+1 = band k
        self.register_buffer("_membership", membership, persistent=False)   # (H*W, K) one-hot
        self.register_buffer("_band_counts", counts, persistent=False)      # (K,)

    # -- band geometry (paper Eqs. 5-7) ------------------------------------------
    def _build_band_geometry(self, h: int, w: int):
        yy = torch.linspace(-1.0, 1.0, h)
        xx = torch.linspace(-1.0, 1.0, w)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        rho = torch.sqrt(grid_x.square() + grid_y.square())
        rho = rho / rho.max()                                                # normalised radius in [0, 1]
        edges = torch.linspace(self.cutoff, 1.0, self.num_bands + 1)
        band = torch.bucketize(rho, edges[1:-1], right=True)                 # 0..K-1; last band is closed at 1.0
        band = torch.where(rho >= self.cutoff, band, torch.full_like(band, -1))
        band_plus1 = (band + 1).flatten()
        membership = F.one_hot(band_plus1, self.num_bands + 1)[:, 1:].float()
        counts = membership.sum(dim=0)
        if bool((counts == 0).any()):
            raise ValueError("At least one radial band is empty; lower NUM_BANDS or the cutoff.")
        return band_plus1, membership, counts

    def _band_descriptor(self, spectrum: torch.Tensor) -> torch.Tensor:
        h, w = spectrum.shape[-2:]
        log_mag = torch.log1p(spectrum.abs() * math.sqrt(h * w)).mean(dim=1)      # mean over channels -> (B, H, W)
        per_band = log_mag.flatten(1) @ self._membership / self._band_counts      # (B, K)
        return per_band / (per_band.mean(dim=1, keepdim=True) + 1e-6)             # within-image normalisation

    def _band_weights(self, spectrum: torch.Tensor) -> torch.Tensor:
        batch = spectrum.shape[0]
        if self.first_stage_mode == "dynamic":
            return self.dynamic_selector(self._band_descriptor(spectrum))
        if self.first_stage_mode == "static":
            return torch.sigmoid(self.static_band_logits).unsqueeze(0).expand(batch, -1)
        return torch.ones(batch, self.num_bands, device=spectrum.device, dtype=spectrum.real.dtype)

    def _band_mask(self, weights: torch.Tensor, h: int, w: int) -> torch.Tensor:
        padded = torch.cat([weights.new_zeros(weights.shape[0], 1), weights], dim=1)  # column 0 -> suppressed
        return padded[:, self._band_plus1].reshape(-1, 1, h, w)

    @torch.no_grad()
    def inspect_stem(self, x: torch.Tensor):
        """(descriptor, weights), both (B, K), for a normalised RGB batch -- for diagnostics only."""
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            spectrum = torch.fft.fftshift(torch.fft.fft2(x.float(), norm="ortho"), dim=[-2, -1])
            return self._band_descriptor(spectrum), self._band_weights(spectrum)

    # main operation of the study
    def _first_stage(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            _, _, h, w = x.shape
            if self.first_stage_mode == "freqnet":
                mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
                mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
                mask = mask[None, None, :, :]
            else:
                mask = self._band_mask(self._band_weights(x), h, w)
            x = x * mask
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self._stem_pending = True   # the next hfreqWH call is the first, image-level one
        return super().forward(x)

    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        if self._stem_pending:
            self._stem_pending = False
            if x.shape[1] != 3:
                raise RuntimeError(f"Expected the 3-channel image at the first stage; got {tuple(x.shape)}")
            return self._first_stage(x, scale)
        return super().hfreqWH(x, scale)


FIRST_STAGE_FOR_MODE = {"dynamic": "dynamic", "static_ring": "static", "ring_ones": "ones"}


class ExperimentModel(nn.Module):
    def __init__(self, input_mode: str = INPUT_MODE):
        super().__init__()
        if input_mode != "rgb_baseline" and input_mode not in FIRST_STAGE_FOR_MODE:
            raise ValueError(f"input_mode must be 'rgb_baseline' or one of {sorted(FIRST_STAGE_FOR_MODE)}")
        self.input_mode = input_mode
        if input_mode == "rgb_baseline":
            self.backbone = AmpSafeFreqNet()
        else:
            self.backbone = DynamicAmpSafeFreqNet(
                first_stage_mode=FIRST_STAGE_FOR_MODE[input_mode], cutoff=MID_HIGH_CUTOFF,
                num_bands=NUM_BANDS, hidden_dim=SELECTOR_HIDDEN, image_size=IMAGE_SIZE,
            )
        self.register_buffer("imagenet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        model_input = (rgb - self.imagenet_mean) / self.imagenet_std
        return self.backbone(model_input).flatten()


RUN_SANITY_CHECK = True


def verify_only_first_stage_differs() -> None:
    with torch.random.fork_rng(devices=list(range(torch.cuda.device_count()))):
        torch.manual_seed(0)
        reference = AmpSafeFreqNet().eval()
        candidate = DynamicAmpSafeFreqNet(first_stage_mode="freqnet", image_size=IMAGE_SIZE).eval()
        candidate.load_state_dict(reference.state_dict(), strict=True)
        probe = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
        with torch.no_grad():
            max_abs_diff = (reference(probe) - candidate(probe)).abs().max().item()
    if max_abs_diff > 1e-6:
        raise AssertionError(f"Dynamic code path does not reproduce original FreqNet (max |diff| = {max_abs_diff:.3e}).")
    print(f"Sanity check passed: original mask through the new code path matches AmpSafeFreqNet (max |diff| = {max_abs_diff:.1e}).")


if RUN_SANITY_CHECK and INPUT_MODE != "rgb_baseline":
    verify_only_first_stage_differs()


model = ExperimentModel(INPUT_MODE)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_grad_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_grad_scaler, init_scale=1024.0, growth_interval=2000)

print("AMP enabled:", use_amp, "| dtype:", amp_dtype if use_amp else "FP32")


# -----------------------------------------------------------------------------
# 5. Training / evaluation -- identical to skripsi__2_.ipynb
# -----------------------------------------------------------------------------


def _forward_and_loss(images: torch.Tensor, labels: torch.Tensor, amp_enabled: bool):
    with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
        logits = model(images)
    loss = criterion(logits.float(), labels.float())
    return logits, loss


def _finite_forward(images: torch.Tensor, labels: torch.Tensor):
    logits, loss = _forward_and_loss(images, labels, amp_enabled=use_amp)
    if torch.isfinite(logits).all() and torch.isfinite(loss):
        return logits, loss, False
    if use_amp:
        logits, loss = _forward_and_loss(images, labels, amp_enabled=False)
        if torch.isfinite(logits).all() and torch.isfinite(loss):
            return logits, loss, True
    bad_logits = int((~torch.isfinite(logits)).sum().item())
    raise FloatingPointError(
        f"Non-finite outputs remain even in a full-FP32 replay: bad_logits={bad_logits}, loss={loss.item()}."
    )


def run_training_epoch(loader: DataLoader) -> float:
    model.train()
    running_loss, seen = 0.0, 0
    fp32_fallback_batches, skipped_gradient_batches = 0, 0
    progress = tqdm(loader, desc="train", leave=False)

    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits, loss, used_fp32_fallback = _finite_forward(images, labels)
        if used_fp32_fallback:
            fp32_fallback_batches += 1

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            if scaler.is_enabled():
                current_scale = scaler.get_scale()
                optimizer.zero_grad(set_to_none=True)
                scaler.update(new_scale=max(current_scale * 0.5, 1.0))
                skipped_gradient_batches += 1
                progress.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}", skipped="grad-overflow")
                continue
            raise FloatingPointError(f"Non-finite gradient norm detected: {grad_norm.item()}")

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        progress.set_postfix(loss=f"{running_loss / seen:.4f}")

    if seen == 0:
        raise RuntimeError("No finite training batches were completed.")
    if fp32_fallback_batches or skipped_gradient_batches:
        print(f"AMP stability: {fp32_fallback_batches} FP32 replays; {skipped_gradient_batches} skipped steps.")
    return running_loss / seen


@torch.inference_mode()
def predict(loader: DataLoader):
    model.eval()
    running_loss, seen = 0.0, 0
    all_labels, all_probabilities, all_paths = [], [], []
    progress = tqdm(loader, desc="evaluate", leave=False)

    for images, labels, paths in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, loss = _forward_and_loss(images, labels, amp_enabled=False)
        if not torch.isfinite(logits).all() or not torch.isfinite(loss):
            raise FloatingPointError("FP32 evaluation produced non-finite values.")
        probabilities = torch.sigmoid(logits.float())
        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        all_labels.append(labels.detach().cpu().numpy().astype(np.int64))
        all_probabilities.append(probabilities.detach().cpu().numpy())
        all_paths.extend(paths)

    return running_loss / seen, np.concatenate(all_labels), np.concatenate(all_probabilities), all_paths

start_epoch = 1
best_val_auc = -np.inf
epochs_without_improvement = 0
history: list[dict[str, float]] = []

# resume mechanism
RESUME_CHECKPOINT_PATH: Path | None = Path('/kaggle/input/datasets/mikeeee54/dynamic25/latest_dynamic_c0.20_seed42_in_progress (2).pt')
LATEST_IN_PROGRESS_PATH = WORK_DIR / f"latest_{RUN_TAG}_in_progress.pt"
if RESUME_CHECKPOINT_PATH is not None and RESUME_CHECKPOINT_PATH.exists():
    print(f"Found an in-progress checkpoint at {RESUME_CHECKPOINT_PATH}, resuming.")
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_auc = checkpoint["best_val_roc_auc_so_far"]
    print(f"Resuming from epoch {start_epoch}, best val ROC-AUC so far: {best_val_auc:.5f}")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n{'=' * 70}\nEpoch {epoch}/{EPOCHS} [{INPUT_MODE}]\n{'=' * 70}")

    train_loss = run_training_epoch(train_loader)
    val_loss, val_y, val_probability, _ = predict(val_loader)
    val_auc = roc_auc_score(val_y, val_probability)
    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_roc_auc": val_auc, "learning_rate": current_lr}
    )
    print(f"Train Loss : {train_loss:.6f}\nVal Loss   : {val_loss:.6f}\nVal ROC-AUC: {val_auc:.6f}\nLR         : {current_lr:.8f}")

    checkpoint_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_roc_auc": val_auc,
        "learning_rate": current_lr,
        "input_mode": INPUT_MODE,
        "mid_high_cutoff": MID_HIGH_CUTOFF,
        "num_bands": NUM_BANDS,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "label_mapping": {"real": 0, "fake": 1},
        "best_val_roc_auc_so_far": max(best_val_auc, val_auc),
    }
    torch.save(checkpoint_payload, LATEST_IN_PROGRESS_PATH)
    torch.save(checkpoint_payload, EPOCH_CHECKPOINT_DIR / f"model_epoch_{epoch:03d}.pt")

    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0
        torch.save({**checkpoint_payload, "best_val_roc_auc": best_val_auc}, CHECKPOINT_PATH)
        print(f"Saved NEW BEST checkpoint: validation ROC-AUC={best_val_auc:.5f}")
    else:
        epochs_without_improvement += 1
        print(f"No ROC-AUC improvement ({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.\nBest validation ROC-AUC: {best_val_auc:.5f}")
        break

history_df = pd.DataFrame(history)
print(history_df)
print(f"\nTraining completed. Best validation ROC-AUC: {best_val_auc:.5f}")
print(f"Best {INPUT_MODE} checkpoint: {CHECKPOINT_PATH}")
history_df.to_csv(WORK_DIR / f"history_{RUN_TAG}.csv", index=False)


DUMP_BAND_WEIGHTS = True


def _unwrap(m: nn.Module) -> nn.Module:
    return m.module if isinstance(m, nn.DataParallel) else m


@torch.inference_mode()
def dump_band_descriptors_and_weights(dataset, out_csv: Path) -> pd.DataFrame:
    net = _unwrap(model).eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rows = []
    for images, labels, paths in tqdm(loader, desc=f"dump {out_csv.stem}", leave=False):
        x = (images.to(device) - net.imagenet_mean) / net.imagenet_std
        descriptor, weights = net.backbone.inspect_stem(x)
        descriptor, weights = descriptor.cpu().numpy(), weights.cpu().numpy()
        for i, path in enumerate(paths):
            row = {"path": path, "label": int(labels[i].item())}
            row.update({f"e{k}": float(descriptor[i, k]) for k in range(NUM_BANDS)})
            row.update({f"w{k}": float(weights[i, k]) for k in range(NUM_BANDS)})
            rows.append(row)
    frame = pd.DataFrame(rows)
    frame.to_csv(out_csv, index=False)
    return frame


if INPUT_MODE != "rgb_baseline" and DUMP_BAND_WEIGHTS:
    best = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    best_state = {k[len("module."):] if k.startswith("module.") else k: v for k, v in best["model_state_dict"].items()}
    _unwrap(model).load_state_dict(best_state)
    weights_df = dump_band_descriptors_and_weights(test_dataset, WORK_DIR / f"band_weights_test_{RUN_TAG}.csv")
    w_cols = [f"w{k}" for k in range(NUM_BANDS)]
    print("\nLearned band weights on the test split (band 0 = lowest retained frequencies)")
    print("mean per band     :", weights_df[w_cols].mean().round(3).tolist())
    print("std across images :", weights_df[w_cols].std().round(3).tolist())
    print("mean by class (0 = real, 1 = fake):")
    print(weights_df.groupby("label")[w_cols].mean().round(3))


Device: cuda
Dataset: /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced
Input mode: dynamic
Checkpoint target: /kaggle/working/best_freqnet_dynamic_c0.20_seed42.pt


Cloning into '/kaggle/working/FreqNet-DeepfakeDetection'...


train      {'real': 19368, 'fake': 19368}
validation {'real': 2767, 'fake': 2767}
test       {'real': 5534, 'fake': 5534}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Sanity check passed: original mask through the new code path matches AmpSafeFreqNet (max |diff| = 0.0e+00).
GPU count: 2
Using 2 GPUs
Trainable parameters: 1,850,985
AMP enabled: True | dtype: torch.bfloat16
Found an in-progress checkpoint at /kaggle/input/datasets/mikeeee54/dynamic25/latest_dynamic_c0.20_seed42_in_progress (2).pt, resuming.
Resuming from epoch 26, best val ROC-AUC so far: 0.99494

Epoch 26/30 [dynamic]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


... (14 outputs left out)

Train Loss : 0.030234
Val Loss   : 0.158526
Val ROC-AUC: 0.995164
LR         : 0.00002500
Saved NEW BEST checkpoint: validation ROC-AUC=0.99516
   epoch  train_loss  val_loss  val_roc_auc  learning_rate
0     26    0.059512  0.177591     0.994380       0.000050
1     27    0.056375  0.199377     0.994053       0.000050
2     28    0.046537  0.254716     0.993507       0.000025
3     29    0.032531  0.181472     0.995042       0.000025
4     30    0.030234  0.158526     0.995164       0.000025

Training completed. Best validation ROC-AUC: 0.99516
Best dynamic checkpoint: /kaggle/working/best_freqnet_dynamic_c0.20_seed42.pt


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


dump band_weights_test_dynamic_c0.20_seed42:   0%|          | 0/692 [00:01<?, ?it/s]


Learned band weights on the test split (band 0 = lowest retained frequencies)
mean per band     : [0.314, 0.572, 0.673, 0.712, 0.694, 0.614, 0.529, 0.364]
std across images : [0.022, 0.034, 0.033, 0.015, 0.033, 0.059, 0.02, 0.03]
mean by class (0 = real, 1 = fake):
          w0     w1     w2     w3     w4     w5     w6     w7
label                                                        
0      0.312  0.570  0.671  0.712  0.695  0.616  0.531  0.366
1      0.316  0.575  0.675  0.712  0.692  0.611  0.528  0.363
